# Append synthetic crops to `JustANormalTinkerer/hayai-synth-dataset`

This notebook generates **new** synthetic Japanese OCR crops and appends them as new parquet shards. It never rewrites or deletes existing shards.

It uses:

- your downloaded **AnimeText background** crops (with `manifest.csv`),
- your **font collection** (matching the paths in `assets/fonts.csv`),
- the optional **real handwritten kana bank** (`manifest.jsonl` + `glyphs/`),
- **lines** from a local CSV/parquet file or a streamed HF dataset.

Each run writes files named:

```
data/<split>-append-<run-id>-00000.parquet
```

which match the dataset card globs (`data/japanese-*.parquet`, `data/sfx-*.parquet`, `data/handwritten-*.parquet`). The schema is identical to the existing dataset:

`image` (JPEG bytes) · `transcription` · `language` · `source` · `image_id` · `vertical`

### Kaggle inputs

Attach these as Kaggle datasets (or set the paths in the config cell):

| Input | What it should contain |
|---|---|
| backgrounds | `manifest.csv` + the `*.png` crops (the output of `generate_backgrounds_animetext`) |
| fonts | the font tree whose relative layout matches `assets/fonts.csv` |
| real kana | a bank folder with `manifest.jsonl` and `glyphs/<kana>/<id>.png` |
| lines (optional) | a `.csv` / `.parquet` with a `line`, `text` or `dialogue` column |

Add your HF token as a Kaggle secret named **`HF_TOKEN`** before enabling upload.


In [ ]:
# ---------------------------------------------------------------------------
# 1. Dependencies
# ---------------------------------------------------------------------------
import subprocess, sys

packages = [
    "datasets>=2.19",
    "huggingface_hub>=0.23",
    "albumentations>=1.1",
    "budoux>=0.1",
    "fire>=0.4",
    "fonttools>=4.30",
    "opencv-python-headless>=4.5",
    "pyarrow>=14",
    "tqdm>=4.60",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("dependencies ready")


In [ ]:
# ---------------------------------------------------------------------------
# 2. Configuration
# ---------------------------------------------------------------------------
# Repository containing this notebook's generator. If you attach the repo as a
# Kaggle dataset, leave REPO_URL empty and the notebook will find it.
REPO_URL = "https://github.com/NopeNopeGuy/hayai-ocr-synth.git"  # @param {type:"string"}
REPO_DIR_OVERRIDE = ""  # @param {type:"string"}

HF_DATASET = "JustANormalTinkerer/hayai-synth-dataset"  # @param {type:"string"}

# Which split to append to. Match this to the crop preset:
#   japanese    -> dialogue crops from novel lines
#   handwritten -> irregular/handwritten crops from dialogue or random lines
#   sfx         -> onomatopoeia crops
SPLIT = "handwritten"  # @param ["japanese", "sfx", "handwritten"]

# What kind of crop to render.
#   auto             = dialogue for japanese, handwritten for handwritten, sfx for sfx
#   dialogue         = normal dialogue, random bubble / no-bubble
#   dialogue_bubble  = force a speech bubble
#   dialogue_flat    = force text directly on the background
#   dialogue_tight   = tight crop around the text, no bubble
#   handwritten      = irregular hand-lettered layout + handwritten fonts + real kana when available
#   sfx              = huge onomatopoeia with curves, outlines and speedlines
#   mixed            = sample from PRESET_MIX
#   custom           = use OVERRIDE_JSON
CROP_PRESET = "handwritten"  # @param ["auto", "dialogue", "dialogue_bubble", "dialogue_flat", "dialogue_tight", "handwritten", "sfx", "mixed", "custom"]

N_IMAGES = 5000  # @param {type:"integer"}
ROWS_PER_SHARD = 5000  # @param {type:"integer"}
WORKERS = 4  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}
JPEG_QUALITY = 95  # @param {type:"integer"}
OUT_DIR = "/kaggle/working/hayai_append"  # @param {type:"string"}
RUN_ID = ""  # @param {type:"string"}
RESUME = False  # @param {type:"boolean"}

# What kind of lines to use.
#   hf_dialogue = stream JustANormalTinkerer/Japanese-Dialogue-Dataset (needs internet)
#   hf_novel    = stream OmniAICreator/Japanese-Novels-23M (needs internet)
#   local       = read LINES_PATH (csv/parquet/jsonl/txt)
#   random      = generator-made random Japanese text (no corpus)
#   sfx         = generator-made onomatopoeia (use with the sfx preset)
LINES_MODE = "hf_dialogue"  # @param ["hf_dialogue", "hf_novel", "local", "random", "sfx"]
LINES_PATH = ""  # @param {type:"string"}
HF_SPLITS = "anime_tv,anime_movie,drama_tv,drama_movie,unsorted"  # @param {type:"string"}
HF_CONFIGS = "Narou_NSFW"  # @param {type:"string"}
TEXT_COLUMN = ""  # @param {type:"string"}
SOURCE_LABEL = ""  # @param {type:"string"}

# Optional explicit input paths. Leave empty to auto-detect under /kaggle/input,
# ~/data and /mnt. All of these are resolved before the generator is imported.
BACKGROUNDS_DIR = ""  # @param {type:"string"}
FONTS_ROOT = ""  # @param {type:"string"}
ASSETS_DIR = ""  # @param {type:"string"}
KANA_BANK_DIR = ""  # @param {type:"string"}

# Crop controls. "auto" keeps the generator's sampling.
VERTICAL = "auto"  # @param ["auto", "true", "false"]
BUBBLE = "auto"  # @param ["auto", "always", "never"]
BUBBLE_PROB = -1.0  # @param {type:"number"}
TIGHT = False  # @param {type:"boolean"}
IRREGULAR = "auto"  # @param ["auto", "true", "false"]
USE_REAL_KANA = "auto"  # @param ["auto", "true", "false"]
REAL_KANA_PROB = -1.0  # @param {type:"number"}
COLOR = "auto"  # @param ["auto", "mono", "color"]
COLOR_MODE = ""  # @param ["", "solid", "per_line", "per_char"]
TEXT_COLOR = ""  # @param {type:"string"}
FONT_SIZE_MIN = 0  # @param {type:"integer"}
FONT_SIZE_MAX = 0  # @param {type:"integer"}
STROKE_SIZE = -1  # @param {type:"integer"}
SLANT_DEG = -999.0  # @param {type:"number"}
DOUBLE_OUTLINE = "auto"  # @param ["auto", "true", "false"]
HARD_INK = "auto"  # @param ["auto", "off"]
PRESET_MIX = "dialogue=0.55,handwritten=0.25,sfx=0.12,dialogue_flat=0.08"  # @param {type:"string"}
OVERRIDE_JSON = ""  # @param {type:"string"}

# Preview before committing to a full run.
PREVIEW_N = 8  # @param {type:"integer"}

# Upload controls. Upload is off by default so a run cannot mutate the Hub by accident.
PUSH_TO_HUB = False  # @param {type:"boolean"}
UPDATE_CARD = True  # @param {type:"boolean"}
DRY_RUN_CARD = True  # @param {type:"boolean"}
VERIFY_REMOTE = False  # @param {type:"boolean"}

print("configuration loaded")


In [ ]:
# ---------------------------------------------------------------------------
# 3. Locate the repository and read the HF token
# ---------------------------------------------------------------------------
import os, shutil, subprocess, sys
from pathlib import Path


def _find_repo():
    candidates = []
    if REPO_DIR_OVERRIDE:
        candidates.append(Path(REPO_DIR_OVERRIDE).expanduser())
    candidates += [
        Path("/kaggle/working/manga-ocr-better-synth"),
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "Github" / "manga-ocr-better-synth",
    ]
    # Search Kaggle inputs to a shallow depth; the repo may be nested one level.
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not root.exists():
            continue
        base = len(root.parts)
        for cur, dirs, files in os.walk(root):
            depth = len(Path(cur).parts) - base
            if depth >= 4:
                dirs[:] = []
            if "tools" in dirs and (Path(cur) / "manga_ocr_dev" / "synthetic_data_generator" / "renderer.py").is_file():
                candidates.append(Path(cur))
            if depth >= 2 and len(candidates) > 40:
                break
        if candidates:
            break
    for path in candidates:
        if (path / "tools" / "append_hayai_synth.py").is_file():
            return path.resolve()
    return None


REPO_DIR = _find_repo()
if REPO_DIR is None:
    if not REPO_URL:
        raise RuntimeError(
            "Could not find the manga-ocr-better-synth repo. Attach it as a Kaggle input "
            "or set REPO_URL."
        )
    target = Path("/kaggle/working/manga-ocr-better-synth")
    if target.exists():
        shutil.rmtree(target)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
    REPO_DIR = target.resolve()

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
SCRIPT = REPO_DIR / "tools" / "append_hayai_synth.py"
if not SCRIPT.is_file():
    raise FileNotFoundError(f"missing {SCRIPT}")

HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or ""

print("repo:", REPO_DIR)
print("script:", SCRIPT)
print("HF token:", "found" if HF_TOKEN else "not set (generation still works, upload will not)")


In [ ]:
# ---------------------------------------------------------------------------
# 4. Auto-discover backgrounds, fonts, kana bank, assets and line files
# ---------------------------------------------------------------------------
import os, re
from pathlib import Path

import pandas as pd


def _iter_dirs(root, max_depth=4):
    root = Path(root)
    base = len(root.parts)
    for cur, dirs, files in os.walk(root):
        depth = len(Path(cur).parts) - base
        if depth >= max_depth:
            dirs[:] = []
        yield Path(cur), files


def _font_sample(fonts_csv, n=600):
    if not Path(fonts_csv).is_file():
        return []
    frame = pd.read_csv(fonts_csv, usecols=["font_path"])
    paths = frame["font_path"].astype(str).tolist()
    if not paths:
        return []
    step = max(1, len(paths) // n)
    return paths[::step][:n]


def _font_score(root, rel_paths):
    root = Path(root)
    if not rel_paths:
        return 0.0
    return sum((root / rel).is_file() for rel in rel_paths) / len(rel_paths)


def _find_assets():
    if ASSETS_DIR:
        return Path(ASSETS_DIR).expanduser()
    env_assets = os.environ.get("MANGA_OCR_ASSETS_PATH")
    if env_assets and (Path(env_assets) / "fonts.csv").is_file():
        return Path(env_assets).expanduser()
    repo_assets = REPO_DIR / "assets"
    if (repo_assets / "fonts.csv").is_file():
        return repo_assets
    for root in (Path("/kaggle/input"), Path("/mnt"), Path.home() / "data"):
        if not root.exists():
            continue
        for cur, files in _iter_dirs(root, max_depth=3):
            if "fonts.csv" in files and "vocab.csv" in files:
                return Path(cur)
    return repo_assets


def _find_fonts_root(assets):
    if FONTS_ROOT:
        return Path(FONTS_ROOT).expanduser()
    env_root = os.environ.get("FONTS_ROOT")
    rel_paths = _font_sample(assets / "fonts.csv")
    if not rel_paths:
        return None
    if env_root and Path(env_root).is_dir() and _font_score(Path(env_root), rel_paths) >= 0.6:
        return Path(env_root).expanduser()

    candidates = set()
    for root in (Path("/kaggle/input"), Path("/mnt"), Path.home() / "data"):
        if not root.exists():
            continue
        for cur, files in _iter_dirs(root, max_depth=5):
            if any(name.lower().endswith((".ttf", ".otf", ".ttc", ".otc")) for name in files):
                path = Path(cur)
                for _ in range(3):
                    candidates.add(path)
                    if path.parent == path or len(path.parts) <= 3:
                        break
                    path = path.parent
    # Also test the explicit env-default just in case it is not under the roots above.
    candidates.add(Path.home() / "data" / "jp_fonts")

    scored = []
    for path in candidates:
        score = _font_score(path, rel_paths)
        if score > 0:
            scored.append((score, -len(path.parts), str(path)))
    scored.sort(reverse=True)
    if scored:
        print("font root candidates (best first):")
        for score, _, path in scored[:5]:
            print(f"  {score:.2f}  {path}")
        if scored[0][0] >= 0.6:
            return Path(scored[0][2])
    return None


def _find_backgrounds():
    if BACKGROUNDS_DIR:
        return Path(BACKGROUNDS_DIR).expanduser()
    env_bg = os.environ.get("BACKGROUND_ANIMETEXT_DIR")
    if env_bg and Path(env_bg).is_dir():
        return Path(env_bg).expanduser()
    best = None
    best_count = 0
    for root in (Path("/kaggle/input"), Path("/mnt"), Path.home() / "data"):
        if not root.exists():
            continue
        for cur, files in _iter_dirs(root, max_depth=4):
            image_like = [
                name
                for name in files
                if name.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
                and re.search(r"_\d+_\d+_\d+_\d+\.(png|jpg|jpeg|webp)$", name, re.I)
            ]
            has_manifest = "manifest.csv" in files
            if image_like or has_manifest:
                count = len(image_like) or len([f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))])
                if count > best_count:
                    best_count = count
                    best = Path(cur)
    return best


def _is_kana_bank(path):
    path = Path(path)
    return (path / "manifest.jsonl").is_file() and (path / "glyphs").is_dir()


def _find_kana_bank():
    if KANA_BANK_DIR:
        return Path(KANA_BANK_DIR).expanduser()
    env_kana = os.environ.get("REAL_KANA_ROOT")
    preferred = [
        Path(env_kana) if env_kana else None,
        Path("/mnt/real-kana-final"),
        Path("/mnt/real-kana-bank"),
        Path.home() / "data" / "real-kana-final",
        REPO_DIR / "assets" / "handwriting",
    ]
    for path in preferred:
        if path is not None and _is_kana_bank(path):
            return path
    found = []
    for root in (Path("/kaggle/input"), Path("/mnt"), Path.home() / "data", REPO_DIR / "assets"):
        if not root.exists():
            continue
        for cur, files in _iter_dirs(root, max_depth=4):
            path = Path(cur)
            if _is_kana_bank(path) and path not in found:
                found.append(path)
    if not found:
        return None
    # Prefer banks with more glyph files when the conventional names are absent.
    return max(found, key=lambda path: sum(1 for _ in (path / "glyphs").glob("*/*.png")))


def _find_lines_file():
    if LINES_PATH:
        return Path(LINES_PATH).expanduser()
    for root in (Path("/kaggle/input"), Path("/mnt"), Path.home() / "data"):
        if not root.exists():
            continue
        for cur, files in _iter_dirs(root, max_depth=4):
            for name in files:
                path = Path(cur) / name
                if path.suffix.lower() not in {".csv", ".parquet"}:
                    continue
                try:
                    if path.suffix.lower() == ".csv":
                        columns = list(pd.read_csv(path, nrows=1).columns)
                    else:
                        import pyarrow.parquet as pq

                        columns = pq.ParquetFile(path).schema_arrow.names
                except Exception:
                    continue
                if any(col in columns for col in ("line", "text", "dialogue", "transcription")):
                    return path
    return None


ASSETS_DIR = str(_find_assets())
FONTS_ROOT = _find_fonts_root(Path(ASSETS_DIR))
BACKGROUNDS_DIR = _find_backgrounds()
KANA_BANK_DIR = _find_kana_bank()
if LINES_MODE == "local" and not LINES_PATH:
    found_lines = _find_lines_file()
    LINES_PATH = str(found_lines) if found_lines else ""

print("assets:      ", ASSETS_DIR)
print("fonts root:  ", FONTS_ROOT)
print("backgrounds: ", BACKGROUNDS_DIR)
if KANA_BANK_DIR:
    print("kana bank:   ", KANA_BANK_DIR)
else:
    print("kana bank:    not found (fonts only)")
if LINES_MODE == "local":
    print("lines file:  ", LINES_PATH or "not found")

if not FONTS_ROOT or not Path(FONTS_ROOT).is_dir():
    raise RuntimeError(
        "Could not find a font root matching assets/fonts.csv. Set FONTS_ROOT in the config "
        "cell to the Kaggle input that preserves the original font directory layout, or run "
        "the optional rescan cell below."
    )
if not BACKGROUNDS_DIR or not Path(BACKGROUNDS_DIR).is_dir():
    raise RuntimeError("Could not find the AnimeText background folder. Set BACKGROUNDS_DIR.")
if LINES_MODE == "local" and not LINES_PATH:
    raise RuntimeError("LINES_MODE='local' but no line CSV/parquet was found. Set LINES_PATH.")


In [ ]:
# ---------------------------------------------------------------------------
# 5. Quick font-root sanity check
# ---------------------------------------------------------------------------
fonts_csv = Path(ASSETS_DIR) / "fonts.csv"
if not fonts_csv.is_file():
    raise FileNotFoundError(
        f"missing {fonts_csv}; set ASSETS_DIR to the folder containing fonts.csv and vocab.csv"
    )
sample = _font_sample(fonts_csv, n=300)
if not sample:
    raise RuntimeError(f"{fonts_csv} is empty; rescan fonts with the optional cell below")
missing = [rel for rel in sample if not (Path(FONTS_ROOT) / rel).is_file()]
print(f"fonts.csv sample: {len(sample) - len(missing)}/{len(sample)} found under {FONTS_ROOT}")
if missing:
    print("first missing:", missing[:3])
if len(missing) > len(sample) * 0.4:
    raise RuntimeError(
        "FONTS_ROOT does not look like the root used to build assets/fonts.csv. "
        "Fix FONTS_ROOT or rescan fonts (see the optional cell at the bottom)."
    )


In [ ]:
# ---------------------------------------------------------------------------
# 6. Build the shared command line
# ---------------------------------------------------------------------------
import shlex
from datetime import datetime, timezone

if not RUN_ID:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-nb"

OUT_DIR = Path(OUT_DIR).expanduser()
if str(OUT_DIR).startswith("/kaggle") and not Path("/kaggle").exists():
    OUT_DIR = Path.cwd() / "hayai_append"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if LINES_MODE == "hf_dialogue":
    LINES_SPEC = "hf:dialogue" + (f":{HF_SPLITS}" if HF_SPLITS else "")
elif LINES_MODE == "hf_novel":
    LINES_SPEC = "hf:novel" + (f":{HF_CONFIGS}" if HF_CONFIGS else "")
elif LINES_MODE == "local":
    LINES_SPEC = str(LINES_PATH)
else:
    LINES_SPEC = LINES_MODE  # random / sfx

COMMON = [
    "--split", SPLIT,
    "--preset", CROP_PRESET,
    "--out-dir", str(OUT_DIR),
    "--run-id", RUN_ID,
    "--lines", LINES_SPEC,
    "--fonts-root", str(FONTS_ROOT),
    "--assets-dir", str(ASSETS_DIR),
    "--backgrounds", str(BACKGROUNDS_DIR),
    "--vertical", VERTICAL,
    "--bubble", BUBBLE,
    "--irregular", IRREGULAR,
    "--color", COLOR,
    "--double-outline", DOUBLE_OUTLINE,
    "--hard-ink", HARD_INK,
    "--real-kana", USE_REAL_KANA,
    "--jpeg-quality", str(JPEG_QUALITY),
    "--seed", str(SEED),
]
if KANA_BANK_DIR:
    COMMON += ["--kana-bank", str(KANA_BANK_DIR)]
if BUBBLE_PROB >= 0:
    COMMON += ["--bubble-prob", str(BUBBLE_PROB)]
if TIGHT:
    COMMON += ["--tight"]
if REAL_KANA_PROB >= 0:
    COMMON += ["--real-kana-prob", str(REAL_KANA_PROB)]
if COLOR_MODE:
    COMMON += ["--color-mode", COLOR_MODE]
if TEXT_COLOR:
    COMMON += ["--text-color", TEXT_COLOR]
if FONT_SIZE_MIN > 0:
    COMMON += ["--font-size-min", str(FONT_SIZE_MIN)]
if FONT_SIZE_MAX > 0:
    COMMON += ["--font-size-max", str(FONT_SIZE_MAX)]
if STROKE_SIZE >= 0:
    COMMON += ["--stroke-size", str(STROKE_SIZE)]
if SLANT_DEG > -900:
    COMMON += ["--slant-deg", str(SLANT_DEG)]
if PRESET_MIX and CROP_PRESET == "mixed":
    COMMON += ["--preset-mix", PRESET_MIX]
if OVERRIDE_JSON:
    COMMON += ["--override-json", OVERRIDE_JSON]
if TEXT_COLUMN:
    COMMON += ["--text-column", TEXT_COLUMN]
if SOURCE_LABEL:
    COMMON += ["--source-label", SOURCE_LABEL]

RUN_INFO = {
    "repo": str(REPO_DIR),
    "dataset": HF_DATASET,
    "split": SPLIT,
    "template": CROP_PRESET,
    "lines": LINES_SPEC,
    "run_id": RUN_ID,
    "n_images": N_IMAGES,
    "rows_per_shard": ROWS_PER_SHARD,
    "seed": SEED,
}
(OUT_DIR / f"run_config-{RUN_ID}.json").write_text(__import__("json").dumps(RUN_INFO, indent=2), encoding="utf-8")
print("run id:", RUN_ID)
print("output:", OUT_DIR)
print("lines: ", LINES_SPEC)
print("\n$", " ".join(shlex.quote(part) for part in [sys.executable, str(SCRIPT), "preview", *COMMON]))


In [ ]:
# ---------------------------------------------------------------------------
# 7. Preview a few crops (runs in one process, no shards written)
# ---------------------------------------------------------------------------
import os, shlex, subprocess, sys
try:
    from IPython.display import Image, display
except Exception:  # local runs without IPython
    Image = None
    display = print

def run_script(args, check=True):
    env = os.environ.copy()
    env.update({
        "FONTS_ROOT": str(FONTS_ROOT),
        "MANGA_OCR_ASSETS_PATH": str(ASSETS_DIR),
        "BACKGROUND_ANIMETEXT_DIR": str(BACKGROUNDS_DIR),
        "OMP_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
    })
    if KANA_BANK_DIR:
        env["REAL_KANA_ROOT"] = str(KANA_BANK_DIR)
    if HF_TOKEN:
        env["HF_TOKEN"] = HF_TOKEN
    cmd = [sys.executable, str(SCRIPT), *args]
    print("$", " ".join(shlex.quote(part) for part in cmd))
    proc = subprocess.run(cmd, env=env)
    if check and proc.returncode:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")
    return proc.returncode

preview_png = OUT_DIR / "preview.png"
run_script([
    "preview", *COMMON,
    "--preview", str(PREVIEW_N),
    "--preview-out", str(preview_png),
    "--workers", "1",
])
if Image is not None:
    display(Image(filename=str(preview_png)))
else:
    print("preview image:", preview_png)


In [ ]:
# ---------------------------------------------------------------------------
# 8. Generate shards
# ---------------------------------------------------------------------------
cmd = [
    "generate", *COMMON,
    "--n-images", str(N_IMAGES),
    "--rows-per-shard", str(ROWS_PER_SHARD),
    "--workers", str(max(1, WORKERS)),
    "--start-method", "fork" if sys.platform.startswith("linux") else "spawn",
]
if RESUME:
    cmd.append("--resume")
run_script(cmd)

import pyarrow.parquet as pq
from pathlib import Path

shards = sorted((OUT_DIR / "data").glob(f"{SPLIT}-append-{RUN_ID}-*.parquet"))
rows = sum(pq.ParquetFile(path).metadata.num_rows for path in shards)
size_gb = sum(path.stat().st_size for path in shards) / (1024 ** 3)
print(f"shards={len(shards)} rows={rows:,} size={size_gb:.2f} GB")
for path in shards:
    print(f"  {path.name}  {pq.ParquetFile(path).metadata.num_rows:,} rows  {path.stat().st_size / 1e6:.1f} MB")


In [ ]:
# ---------------------------------------------------------------------------
# 9. Validate local shards (schema, JPEG magic, labels, unique ids)
# ---------------------------------------------------------------------------
run_script(["validate", "--out-dir", str(OUT_DIR), "--split", SPLIT, "--run-id", RUN_ID])


In [ ]:
# ---------------------------------------------------------------------------
# 10. Upload only the new shards
# ---------------------------------------------------------------------------
ROWS_TO_ADD = 0
NEW_SHARDS = []
if not PUSH_TO_HUB:
    print("PUSH_TO_HUB is False; nothing uploaded.")
    print("Local shards are ready at:", OUT_DIR / "data")
else:
    if not HF_TOKEN:
        raise RuntimeError(
            "PUSH_TO_HUB is True but no HF_TOKEN was found. Add a Kaggle secret named HF_TOKEN."
        )
    from huggingface_hub import HfApi

    api = HfApi(token=HF_TOKEN)
    remote = set(api.list_repo_files(HF_DATASET, repo_type="dataset"))
    NEW_SHARDS = [p for p in shards if f"data/{p.name}" not in remote]
    ROWS_TO_ADD = sum(pq.ParquetFile(p).metadata.num_rows for p in NEW_SHARDS)
    print(f"new shards: {len(NEW_SHARDS)} ({ROWS_TO_ADD:,} rows); already present: {len(shards) - len(NEW_SHARDS)}")
    if NEW_SHARDS:
        run_script(["upload", "--out-dir", str(OUT_DIR), "--split", SPLIT, "--run-id", RUN_ID, "--repo-id", HF_DATASET])
    else:
        print("Nothing new to upload (all shards already exist in the repo).")


In [ ]:
# ---------------------------------------------------------------------------
# 11. Update README split counts
# ---------------------------------------------------------------------------
# The card stores explicit row counts per split. This adds only the rows that
# were actually uploaded in this run. DRY_RUN_CARD=True prints the result first.
if PUSH_TO_HUB and UPDATE_CARD and ROWS_TO_ADD:
    card_cmd = [
        "update-card",
        "--repo-id", HF_DATASET,
        "--add", f"{SPLIT}={ROWS_TO_ADD}",
        "--wait", "300",
    ]
    if DRY_RUN_CARD:
        card_cmd.append("--dry-run")
    run_script(card_cmd)
elif PUSH_TO_HUB and UPDATE_CARD:
    print("No newly uploaded rows; card not changed.")
else:
    print("Card update skipped.")


In [ ]:
# ---------------------------------------------------------------------------
# 12. Optional remote spot-check
# ---------------------------------------------------------------------------
if VERIFY_REMOTE and NEW_SHARDS:
    from datasets import load_dataset

    sample_file = NEW_SHARDS[-1].name
    url = f"https://huggingface.co/datasets/{HF_DATASET}/resolve/main/data/{sample_file}"
    ds = load_dataset("parquet", data_files=url, split="train")
    print(ds)
    row = ds[0]
    print(repr(row["transcription"]), row["language"], row["source"], row["vertical"])
    display(row["image"])
else:
    print("Remote check skipped (set VERIFY_REMOTE=True after uploading).")


In [ ]:
# ---------------------------------------------------------------------------
# Optional: rescan fonts if FONTS_ROOT has no matching assets/fonts.csv
# ---------------------------------------------------------------------------
# Only run this if the sanity check above fails. Scanning ~20k fonts can take
# hours. The scan writes into a writable copy of assets under /kaggle/working.
RESCAN_FONTS = False  # @param {type:"boolean"}
if RESCAN_FONTS:
    import shutil

    work_assets = Path("/kaggle/working/assets")
    if work_assets.exists():
        shutil.rmtree(work_assets)
    shutil.copytree(ASSETS_DIR, work_assets)
    env = os.environ.copy()
    env.update({
        "FONTS_ROOT": str(FONTS_ROOT),
        "MANGA_OCR_ASSETS_PATH": str(work_assets),
        "OMP_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
    })
    cmd = [sys.executable, "-m", "manga_ocr_dev.synthetic_data_generator.scan_fonts", "--max_workers", str(max(1, WORKERS))]
    print("$", " ".join(cmd))
    subprocess.run(cmd, env=env, check=True)
    ASSETS_DIR = str(work_assets)
    print("new assets:", ASSETS_DIR)


## Notes and troubleshooting

### Crop presets

| Preset | What you get |
|---|---|
| `dialogue` | normal dialogue; bubble / no-bubble is sampled |
| `dialogue_bubble` | forces a speech bubble |
| `dialogue_flat` | text directly on the background |
| `dialogue_tight` | tight crop around the text, no bubble |
| `handwritten` | irregular layout, handwritten fonts, real kana when a bank is attached |
| `sfx` | onomatopoeia with curves, outlines, speedlines and tight crops |
| `mixed` | weighted mix from `PRESET_MIX` |
| `custom` | merge your own `OVERRIDE_JSON` |

The generator's full override vocabulary is available through `OVERRIDE_JSON`,
for example:

```json
{"font_size": 44, "stroke_size": 3, "letter_spacing": 0.05, "slant": -8}
```

### Line sources

- `hf_dialogue` streams `JustANormalTinkerer/Japanese-Dialogue-Dataset` and yields
  sources such as `dialogue_anime_tv`, matching the existing `handwritten` split.
- `hf_novel` streams `OmniAICreator/Japanese-Novels-23M`; use `Narou_NSFW` to
  match the existing `japanese` split (`source=novel_Narou_NSFW`).
- `local` reads CSV/parquet/JSONL/TXT. A `source` column is preserved; otherwise
  the file stem becomes the source.
- `random` and `sfx` need no corpus. They are useful for smoke tests but produce
  synthetic word salad, not natural lines.

### Why new file names?

The repo already contains `data/handwritten-00000-of-00021.parquet`. Reusing
those names would overwrite real data. This notebook writes
`data/handwritten-append-<run-id>-00000.parquet`, which still matches the card's
`data/handwritten-*.parquet` glob. Uploading is additive; nothing is deleted.

### If font discovery fails

`assets/fonts.csv` stores font paths relative to the original root. Your Kaggle
font dataset must preserve that relative layout. If it does not, run the
optional rescan cell (slow) or regenerate `fonts.csv` locally and attach it
alongside the fonts.

### If HF streaming is unavailable

Kaggle notebooks can run with internet disabled. In that case set
`LINES_MODE = "local"` and attach a CSV/parquet with a `line` column, or use
`random` / `sfx`.
